In [1]:
import requests
import re
import json
import pandas as pd
import time
from urllib.parse import urlparse
from bs4 import BeautifulSoup
from dotenv import load_dotenv
import os

In [2]:
# 1. Obtener el HTML de la página del ranking
load_dotenv()
url = os.getenv("TARGET_URL_CPI")
BASE_URL = os.getenv("BASE_URL")

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)
response.raise_for_status()  # Lanza error si la petición falla

In [3]:
# 2. Parsear HTML con BeautifulSoup
soup = BeautifulSoup(response.text, "html.parser")

In [4]:
# 3. Buscar la tabla principal y extraer los links
table = soup.find("table")
country_links = []


if table:
    rows = table.find_all("tr")
    for row in rows:
        cell = row.find("td")
        if cell and cell.a:
            relative_link = cell.a["href"]
            full_url = f"{BASE_URL}{relative_link}"
            country_links.append(full_url)

# Convertir la lista en DataFrame
df_links = pd.DataFrame(country_links, columns=["url"])

# Guardar como archivo Parquet
df_links.to_parquet("../../data/country_cpi_links.parquet", index=False)

In [5]:
# 4. Mostrar resultado
print("Links encontrados:", df_links.size)

Links encontrados: 182


In [6]:
# configuracion necesaria para descargar datos

master_df = pd.DataFrame()

pattern = r'arrayToDataTable\(\s*(\[\[.*?\]\])\s*\);'

sleep_duration = 5

In [7]:
master_df.head()

""


In [8]:
# Cargar los países válidos desde el archivo parquet
df_countries = pd.read_parquet("../../data/countries_with_hpi.parquet")
valid_countries = set(df_countries["country"].unique())

In [9]:
# DataFrames de salida
dfs_quarters = []
dfs_months = []

start_time = time.time()
for url in country_links:
    country = urlparse(url).path.strip("/").split("/")[0]

    if country not in valid_countries:
        print(f"⏭️ País {country} no está en la lista, se omite.")
        continue

    try:
        response = requests.get(url, headers=headers)
        html = response.text

        match = re.search(pattern, html, re.DOTALL)
        if match:
            data_str = match.group(1)
            data = json.loads(data_str)

            df = pd.DataFrame(data[1:], columns=data[0])
            df.columns = ["Period", "Value"]
            df["Country"] = country

            # --- separar según formato de Period ---
            if df["Period"].str.contains(r"Q\d").any():
                df_pivot = df.pivot(index="Period", columns="Country", values="Value")
                dfs_quarters.append(df_pivot)
                print(f"✅ Datos trimestrales extraídos para {country}")
            else:
                df_pivot = df.pivot(index="Period", columns="Country", values="Value")
                dfs_months.append(df_pivot)
                print(f"✅ Datos mensuales extraídos para {country}")
        else:
            print(f"⚠️ No se encontraron datos en {url}")
    except Exception as e:
        print(f"❌ Error procesando {url}: {e}")
    
    time.sleep(sleep_duration)

# unir cada grupo por separado
df_quarters = pd.concat(dfs_quarters, axis=1) if dfs_quarters else pd.DataFrame()
df_months = pd.concat(dfs_months, axis=1) if dfs_months else pd.DataFrame()

end_time = time.time()
total_sleep_time = len(valid_countries) * sleep_duration
effective_time = end_time - start_time - total_sleep_time

print(f"⏳ Tiempo total (incluyendo esperas): {end_time - start_time:.2f} s")
print(f"😴 Tiempo en espera: {total_sleep_time:.2f} s")
print(f"⚡ Tiempo efectivo de ejecución: {effective_time:.2f} s")

⏭️ País Afghanistan no está en la lista, se omite.
⏭️ País Albania no está en la lista, se omite.
⏭️ País Algeria no está en la lista, se omite.
⏭️ País Andorra no está en la lista, se omite.
⏭️ País Angola no está en la lista, se omite.
⏭️ País Antigua-and-Barbuda no está en la lista, se omite.
⏭️ País Argentina no está en la lista, se omite.
⏭️ País Armenia no está en la lista, se omite.
⏭️ País Aruba no está en la lista, se omite.
✅ Datos trimestrales extraídos para Australia
✅ Datos mensuales extraídos para Austria
⏭️ País Azerbaijan no está en la lista, se omite.
⏭️ País Bahamas no está en la lista, se omite.
⏭️ País Bahrain no está en la lista, se omite.
⏭️ País Bangladesh no está en la lista, se omite.
⏭️ País Barbados no está en la lista, se omite.
⏭️ País Belarus no está en la lista, se omite.
✅ Datos mensuales extraídos para Belgium
⏭️ País Belize no está en la lista, se omite.
⏭️ País Benin no está en la lista, se omite.
⏭️ País Bermuda no está en la lista, se omite.
⏭️ País

In [10]:
# resultados finales
print("\n📊 DataFrame trimestral")
df_quarters.head()


📊 DataFrame trimestral


Country,Australia,New-Zealand
Period,,
Q1 1960,7.5,44.8764
Q1 1961,7.8,45.5819
Q1 1962,7.8,46.9145
Q1 1963,7.8,47.7767
Q1 1964,8.0,48.9917


In [30]:
print("\n📊 DataFrame mensual")
df_months.head()


📊 DataFrame mensual


Country,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,China,Colombia,Croatia,Cyprus,...,Slovenia,South-Africa,South-Korea,Spain,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom
Period,,,,,,,,,,,,,,,,,,,,,
Apr 1996,71.69,70.47,1298.65,NaN,88.5,60.405,80.6937,24.21,NaN,66.76,...,45.76,NaN,54.226,64.73,76.49,NaN,60.2458,1.42,156.1,68.7
Apr 1997,72.56,71.25,1405.18,39.84,90.2,64.062,83.2759,28.69,NaN,69.43,...,48.92,NaN,56.565,65.77,77.39,NaN,62.7807,2.50,159.9,69.8
Apr 1998,73.42,72.16,1459.71,49.05,91.1,67.511,83.0260,34.65,63.74,70.47,...,53.37,NaN,61.557,67.01,78.49,NaN,69.0334,4.80,162.2,71.1
Apr 1999,73.49,72.94,1508.55,47.95,92.0,70.234,81.1995,38.51,65.96,70.83,...,55.80,NaN,61.816,68.56,78.74,NaN,69.3714,7.76,165.9,72.2
Apr 2000,74.78,74.63,1610.75,52.36,94.8,72.715,80.9559,42.35,68.53,74.98,...,60.98,NaN,62.709,70.64,79.49,NaN,69.7000,12.52,170.9,72.6


---

In [12]:
# obtain paraguay url
template_url = country_links[0]
country = "Paraguay"

# Separar por "/"
parts = template_url.strip("/").split("/")

# Reemplazar el país (posición 3: después del dominio)

parts[3] = country

# Reconstruir la URL
custom_url = "/".join(parts) + "/"

In [13]:
# URL específica para Paraguay
response = requests.get(custom_url, headers=headers)
html = response.text

# Buscar el bloque JSON dentro del HTML
match = re.search(pattern, html, re.DOTALL)
if match:
    data_str = match.group(1)
    data = json.loads(data_str)

    # Crear DataFrame con los datos
    df = pd.DataFrame(data[1:], columns=data[0])
    df.columns = ["Period", "Value"]
    df["Country"] = "Paraguay"  # Asignar explícitamente

    # Pivotear para formato ancho (opcional)
    df_paraguay = df.pivot(index="Period", columns="Country", values="Value")

    print("✅ Datos de Paraguay extraídos correctamente")
else:
    print("⚠️ No se encontraron datos en el HTML para Paraguay")

✅ Datos de Paraguay extraídos correctamente


In [32]:
df_paraguay.head()

Country,Paraguay
Period,
Apr 1995,22.2615
Apr 1996,24.6815
Apr 1997,26.6891
Apr 1998,28.9441
Apr 1999,31.1579


In [15]:
# Unir ambos DataFrames por el índice 'Period', preservando todos los datos
new_month_df = pd.merge(df_months, df_paraguay, how="outer", left_index=True, right_index=True)

In [31]:
new_month_df.head()

Country,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,China,Colombia,Croatia,Cyprus,...,South-Africa,South-Korea,Spain,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom,Paraguay
Period,,,,,,,,,,,,,,,,,,,,,
Apr 1960,NaN,NaN,NaN,NaN,15.5,NaN,NaN,0.0401,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29.54,NaN,NaN
Apr 1961,NaN,NaN,NaN,NaN,15.7,NaN,NaN,0.0440,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29.81,NaN,NaN
Apr 1962,NaN,NaN,NaN,NaN,15.7,NaN,NaN,0.0456,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.21,NaN,NaN
Apr 1963,NaN,NaN,NaN,NaN,16.0,NaN,NaN,0.0571,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.48,NaN,NaN
Apr 1964,NaN,NaN,NaN,NaN,16.3,NaN,NaN,0.0674,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.95,NaN,NaN


In [25]:
unique_periods = pd.Series(new_month_df.index.astype(str)).unique()
print(unique_periods)

['Apr 1960' 'Apr 1961' 'Apr 1962' 'Apr 1963' 'Apr 1964' 'Apr 1965'
 'Apr 1966' 'Apr 1967' 'Apr 1968' 'Apr 1969' 'Apr 1970' 'Apr 1971'
 'Apr 1972' 'Apr 1973' 'Apr 1974' 'Apr 1975' 'Apr 1976' 'Apr 1977'
 'Apr 1978' 'Apr 1979' 'Apr 1980' 'Apr 1981' 'Apr 1982' 'Apr 1983'
 'Apr 1984' 'Apr 1985' 'Apr 1986' 'Apr 1987' 'Apr 1988' 'Apr 1989'
 'Apr 1990' 'Apr 1991' 'Apr 1992' 'Apr 1993' 'Apr 1994' 'Apr 1995'
 'Apr 1996' 'Apr 1997' 'Apr 1998' 'Apr 1999' 'Apr 2000' 'Apr 2001'
 'Apr 2002' 'Apr 2003' 'Apr 2004' 'Apr 2005' 'Apr 2006' 'Apr 2007'
 'Apr 2008' 'Apr 2009' 'Apr 2010' 'Apr 2011' 'Apr 2012' 'Apr 2013'
 'Apr 2014' 'Apr 2015' 'Apr 2016' 'Apr 2017' 'Apr 2018' 'Apr 2019'
 'Apr 2020' 'Apr 2021' 'Apr 2022' 'Apr 2023' 'Apr 2024' 'Apr 2025'
 'Aug 1960' 'Aug 1961' 'Aug 1962' 'Aug 1963' 'Aug 1964' 'Aug 1965'
 'Aug 1966' 'Aug 1967' 'Aug 1968' 'Aug 1969' 'Aug 1970' 'Aug 1971'
 'Aug 1972' 'Aug 1973' 'Aug 1974' 'Aug 1975' 'Aug 1976' 'Aug 1977'
 'Aug 1978' 'Aug 1979' 'Aug 1980' 'Aug 1981' 'Aug 1982' 'Aug 1

In [26]:
unique_periods = pd.Series(df_quarters.index.astype(str)).unique()
print(unique_periods)

['Q1 1960' 'Q1 1961' 'Q1 1962' 'Q1 1963' 'Q1 1964' 'Q1 1965' 'Q1 1966'
 'Q1 1967' 'Q1 1968' 'Q1 1969' 'Q1 1970' 'Q1 1971' 'Q1 1972' 'Q1 1973'
 'Q1 1974' 'Q1 1975' 'Q1 1976' 'Q1 1977' 'Q1 1978' 'Q1 1979' 'Q1 1980'
 'Q1 1981' 'Q1 1982' 'Q1 1983' 'Q1 1984' 'Q1 1985' 'Q1 1986' 'Q1 1987'
 'Q1 1988' 'Q1 1989' 'Q1 1990' 'Q1 1991' 'Q1 1992' 'Q1 1993' 'Q1 1994'
 'Q1 1995' 'Q1 1996' 'Q1 1997' 'Q1 1998' 'Q1 1999' 'Q1 2000' 'Q1 2001'
 'Q1 2002' 'Q1 2003' 'Q1 2004' 'Q1 2005' 'Q1 2006' 'Q1 2007' 'Q1 2008'
 'Q1 2009' 'Q1 2010' 'Q1 2011' 'Q1 2012' 'Q1 2013' 'Q1 2014' 'Q1 2015'
 'Q1 2016' 'Q1 2017' 'Q1 2018' 'Q1 2019' 'Q1 2020' 'Q1 2021' 'Q1 2022'
 'Q1 2023' 'Q1 2024' 'Q1 2025' 'Q2 1960' 'Q2 1961' 'Q2 1962' 'Q2 1963'
 'Q2 1964' 'Q2 1965' 'Q2 1966' 'Q2 1967' 'Q2 1968' 'Q2 1969' 'Q2 1970'
 'Q2 1971' 'Q2 1972' 'Q2 1973' 'Q2 1974' 'Q2 1975' 'Q2 1976' 'Q2 1977'
 'Q2 1978' 'Q2 1979' 'Q2 1980' 'Q2 1981' 'Q2 1982' 'Q2 1983' 'Q2 1984'
 'Q2 1985' 'Q2 1986' 'Q2 1987' 'Q2 1988' 'Q2 1989' 'Q2 1990' 'Q2 1991'
 'Q2 1

In [33]:
def split_period_index(df):
    """
    Recibe un DataFrame con índice 'Period' (ej: 'Apr 1960' o 'Q1 1990')
    Devuelve el mismo DataFrame con columnas 'Year' y 'Month' o 'Quarter'
    """
    df = df.copy().reset_index()  # sacar Period del índice
    df["Year"] = None
    df["Month"] = None
    df["Quarter"] = None

    for i, period in enumerate(df["Period"]):
        parts = period.split()

        if len(parts) == 2:
            first, year = parts
            df.at[i, "Year"] = int(year)

            if "Q" in first:  # Formato trimestral
                df.at[i, "Quarter"] = first
            else:  # Formato mensual
                df.at[i, "Month"] = first

    return df.set_index("Period")  # opcional: devolver con Period otra vez de índice


In [34]:
new_month_df = split_period_index(new_month_df)

In [35]:
new_month_df.head()

Country,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,China,Colombia,Croatia,Cyprus,...,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom,Paraguay,Year,Month,Quarter
Period,,,,,,,,,,,,,,,,,,,,,
Apr 1960,NaN,NaN,NaN,NaN,15.5,NaN,NaN,0.0401,NaN,NaN,...,NaN,NaN,NaN,NaN,29.54,NaN,NaN,1960,Apr,None
Apr 1961,NaN,NaN,NaN,NaN,15.7,NaN,NaN,0.0440,NaN,NaN,...,NaN,NaN,NaN,NaN,29.81,NaN,NaN,1961,Apr,None
Apr 1962,NaN,NaN,NaN,NaN,15.7,NaN,NaN,0.0456,NaN,NaN,...,NaN,NaN,NaN,NaN,30.21,NaN,NaN,1962,Apr,None
Apr 1963,NaN,NaN,NaN,NaN,16.0,NaN,NaN,0.0571,NaN,NaN,...,NaN,NaN,NaN,NaN,30.48,NaN,NaN,1963,Apr,None
Apr 1964,NaN,NaN,NaN,NaN,16.3,NaN,NaN,0.0674,NaN,NaN,...,NaN,NaN,NaN,NaN,30.95,NaN,NaN,1964,Apr,None


In [36]:
new_month_df.tail()

Country,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,China,Colombia,Croatia,Cyprus,...,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom,Paraguay,Year,Month,Quarter
Period,,,,,,,,,,,,,,,,,,,,,
Sep 2020,108.67,108.21,5391.75,105.70,136.9,79.72,127.9984,105.29,103.18,99.70,...,108.06,100.44,102.18,182.90,260.280,109.1,106.6,2020,Sep,None
Sep 2021,112.23,112.35,5944.21,109.92,142.9,83.98,128.4814,110.04,106.80,103.31,...,111.29,101.29,101.21,218.71,274.310,112.4,113.4,2021,Sep,None
Sep 2022,124.58,125.90,6370.34,127.10,152.7,95.51,131.9948,122.63,120.21,112.63,...,122.70,104.53,107.70,401.10,296.808,122.3,124.0,2022,Sep,None
Sep 2023,131.75,126.73,6700.66,135.18,158.5,100.39,132.1184,136.11,129.15,117.48,...,127.19,106.57,108.02,648.45,307.789,130.1,128.3,2023,Sep,None
Sep 2024,134.13,132.21,6997.15,137.25,161.1,104.54,132.7618,144.02,133.11,119.41,...,128.76,107.50,108.68,969.11,315.301,133.5,133.5,2024,Sep,None


In [37]:
new_quarter_df = split_period_index(df_quarters)
new_quarter_df.head()

Country,Australia,New-Zealand,Year,Month,Quarter
Period,,,,,
Q1 1960,7.5,44.8764,1960,None,Q1
Q1 1961,7.8,45.5819,1961,None,Q1
Q1 1962,7.8,46.9145,1962,None,Q1
Q1 1963,7.8,47.7767,1963,None,Q1
Q1 1964,8.0,48.9917,1964,None,Q1


In [43]:
def month_to_quarter(df):
    """
    Takes a DataFrame with 'Month' and 'Quarter' columns.
    If 'Month' exists, it assigns the corresponding quarter to 'Quarter'.
    """
    df = df.copy()

    # Dictionary for mapping months → quarters
    month_to_q = {
        "Jan": 1, "Feb": 1, "Mar": 1,
        "Apr": 2, "May": 2, "June": 2,
        "Jul": 3, "Aug": 3, "Sep": 3,
        "Oct": 4, "Nov": 4, "Dec": 4,
    }

    # Update quarter only where Month exists
    df["Quarter"] = df.apply(
        lambda row: month_to_q.get(row["Month"], row["Quarter"]),
        axis=1
    )

    return df

In [44]:
new_month_df = month_to_quarter(new_month_df)

In [45]:
new_month_df.tail()

Country,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,China,Colombia,Croatia,Cyprus,...,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom,Paraguay,Year,Month,Quarter
Period,,,,,,,,,,,,,,,,,,,,,
Sep 2020,108.67,108.21,5391.75,105.70,136.9,79.72,127.9984,105.29,103.18,99.70,...,108.06,100.44,102.18,182.90,260.280,109.1,106.6,2020,Sep,3
Sep 2021,112.23,112.35,5944.21,109.92,142.9,83.98,128.4814,110.04,106.80,103.31,...,111.29,101.29,101.21,218.71,274.310,112.4,113.4,2021,Sep,3
Sep 2022,124.58,125.90,6370.34,127.10,152.7,95.51,131.9948,122.63,120.21,112.63,...,122.70,104.53,107.70,401.10,296.808,122.3,124.0,2022,Sep,3
Sep 2023,131.75,126.73,6700.66,135.18,158.5,100.39,132.1184,136.11,129.15,117.48,...,127.19,106.57,108.02,648.45,307.789,130.1,128.3,2023,Sep,3
Sep 2024,134.13,132.21,6997.15,137.25,161.1,104.54,132.7618,144.02,133.11,119.41,...,128.76,107.50,108.68,969.11,315.301,133.5,133.5,2024,Sep,3


In [46]:
def quarter_to_int(df):
    """
    Convierte los valores de la columna 'Quarter' a enteros (Q1 → 1, Q2 → 2, etc.)
    """
    df = df.copy()

    q_map = {"Q1": 1, "Q2": 2, "Q3": 3, "Q4": 4}

    df["Quarter"] = df["Quarter"].map(q_map).fillna(df["Quarter"])

    return df

In [47]:
new_quarter_df = quarter_to_int(new_quarter_df)
new_quarter_df.head()

Country,Australia,New-Zealand,Year,Month,Quarter
Period,,,,,
Q1 1960,7.5,44.8764,1960,None,1
Q1 1961,7.8,45.5819,1961,None,1
Q1 1962,7.8,46.9145,1962,None,1
Q1 1963,7.8,47.7767,1963,None,1
Q1 1964,8.0,48.9917,1964,None,1


In [48]:
new_quarter_df.dtypes

Country
Australia      float64
New-Zealand    float64
Year            object
Month           object
Quarter          int64
dtype: object

In [49]:
def year_to_int(df):
    """
    Convierte la columna 'Year' a enteros.
    Si hay valores no numéricos, los convierte en NaN.
    """
    df = df.copy()
    df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")
    return df

In [50]:
new_quarter_df = year_to_int(new_quarter_df)
new_quarter_df.dtypes

Country
Australia      float64
New-Zealand    float64
Year             Int64
Month           object
Quarter          int64
dtype: object

In [51]:
new_quarter_df.head()

Country,Australia,New-Zealand,Year,Month,Quarter
Period,,,,,
Q1 1960,7.5,44.8764,1960,None,1
Q1 1961,7.8,45.5819,1961,None,1
Q1 1962,7.8,46.9145,1962,None,1
Q1 1963,7.8,47.7767,1963,None,1
Q1 1964,8.0,48.9917,1964,None,1


In [52]:
new_month_df = year_to_int(new_month_df)
new_month_df.dtypes

Country
Austria           float64
Belgium           float64
Brazil            float64
Bulgaria          float64
Canada            float64
Chile             float64
China             float64
Colombia          float64
Croatia           float64
Cyprus            float64
Czech-Republic    float64
Denmark           float64
Estonia           float64
Finland           float64
France            float64
Germany           float64
Greece            float64
Hong-Kong         float64
Hungary           float64
Iceland           float64
India             float64
Indonesia         float64
Ireland           float64
Israel            float64
Italy             float64
Japan             float64
Latvia            float64
Lithuania         float64
Luxembourg        float64
Macedonia         float64
Malaysia          float64
Malta             float64
Mexico            float64
Morocco           float64
Netherlands       float64
Norway            float64
Peru              float64
Philippines       float64
Pola

In [55]:
new_month_df.tail()

Country,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,China,Colombia,Croatia,Cyprus,...,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom,Paraguay,Year,Month,Quarter
Period,,,,,,,,,,,,,,,,,,,,,
Sep 2020,108.67,108.21,5391.75,105.70,136.9,79.72,127.9984,105.29,103.18,99.70,...,108.06,100.44,102.18,182.90,260.280,109.1,106.6,2020,Sep,3
Sep 2021,112.23,112.35,5944.21,109.92,142.9,83.98,128.4814,110.04,106.80,103.31,...,111.29,101.29,101.21,218.71,274.310,112.4,113.4,2021,Sep,3
Sep 2022,124.58,125.90,6370.34,127.10,152.7,95.51,131.9948,122.63,120.21,112.63,...,122.70,104.53,107.70,401.10,296.808,122.3,124.0,2022,Sep,3
Sep 2023,131.75,126.73,6700.66,135.18,158.5,100.39,132.1184,136.11,129.15,117.48,...,127.19,106.57,108.02,648.45,307.789,130.1,128.3,2023,Sep,3
Sep 2024,134.13,132.21,6997.15,137.25,161.1,104.54,132.7618,144.02,133.11,119.41,...,128.76,107.50,108.68,969.11,315.301,133.5,133.5,2024,Sep,3


In [56]:
# save as parquet file
new_quarter_df.to_parquet("../../data/quarter_cpi_values.parquet", index=False)

In [58]:
# save as parquet file
new_month_df.to_parquet("../../data/month_cpi_values.parquet", index=False)